In [1]:
import os
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')
curr_path = os.path.join(os.getcwd(), 'drive', 'MyDrive', 'Colab Notebooks', 'Satellite')
DATASETS_Main = ()
LABLES_Main = ()

Mounted at /content/drive


In [ ]:
# print(curr_path)
# asd = os.path.join(curr_path, 'Dataset_LC08_L1TP_098084_20250207_20250214_02_T1.npy')
# dataset1 = np.load(asd);#, mmap_mode=None, allow_pickle=True, fix_imports=True, encoding='ASCII')

/content/drive/MyDrive/Colab Notebooks/Satellite


In [ ]:
# list_ds = [i for i in os.listdir(curr_path) if 'Dataset' in i]
# list_lbl = [i for i in os.listdir(curr_path) if 'Labels' in i]
# np.size(list_lbl)

23

In [3]:
#Здесь должна быть функция, которая конкатенирует свежеподгруженные датасеты и лейблы со старыми
def _Concat(lsd, lsl):
  DATASETS = np.concatenate([np.load(os.path.join(curr_path, i)) for i in lsd], axis = 0)
  LABLES = np.concatenate([np.load(os.path.join(curr_path, i)) for i in lsl], axis = 0)
  # print(DATASETS.shape)
  # print(LABLES.shape)
  return DATASETS, LABLES

In [4]:
#Автоматизировать обращение к базе данных
#Начать делать презентацию
#
#Актуальность: наша цель научить нейросеть распознавать объекты на Земле (автоматическая геообработка). Мы опираемся на аналитику, но она неточная, учитывая это
#Наша акутальность не слишком понятно выглядит. Актуальность в том, чтобы научить нейросеть определять лейблы так как определяем её мы, точно определять объекты
#на Земле.

In [5]:
def getData():
  asd = os.path.join(curr_path, 'Dataset_LC08_L1TP_098084_20250207_20250214_02_T1.npy')
  dataset1 = np.load(asd);#, mmap_mode=None, allow_pickle=True, fix_imports=True, encoding='ASCII')
  list_ds = [i for i in os.listdir(curr_path) if 'Dataset' in i]
  list_lbl = [i for i in os.listdir(curr_path) if 'Labels' in i]
  return _Concat(list_ds, list_lbl)

In [6]:
(DATASETS_Main, LABLES_Main) = getData();

In [8]:
print(DATASETS_Main.shape)
print(LABLES_Main.shape)

(147, 11, 500, 500)
(147, 5, 500, 500)


In [14]:
# Импорт всех необходимых библиотек
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D, concatenate, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Загрузка данных (замените пути на свои!)
# Предполагаем, что данные уже загружены в переменные DATASETS_Main и LABLES_Main
# DATASETS_Main.shape = (147, 11, 500, 500) - 147 изображений, 11 каналов, размер 500x500
# LABLES_Main.shape = (147, 5, 500, 500) - 147 масок, 5 классов, размер 500x500

# Пример создания случайных данных для демонстрации (удалите эти строки, если у вас реальные данные!)
DATASETS_Main = np.random.rand(147, 11, 500, 500).astype(np.float32)
LABLES_Main = np.random.randint(0, 5, size=(147, 5, 500, 500)).astype(np.float32)

# Преобразование данных в формат, который понимает TensorFlow (изменение порядка осей)
# TensorFlow ожидает формат (batch, height, width, channels)
X = np.moveaxis(DATASETS_Main, 1, -1)  # Теперь shape: (147, 500, 500, 11)
y = np.moveaxis(LABLES_Main, 1, -1)    # Теперь shape: (147, 500, 500, 5)

# Разделение данных на обучающую, валидационную и тестовую выборки (70%/15%/15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Обучающая выборка: {X_train.shape}")
print(f"Валидационная выборка: {X_val.shape}")
print(f"Тестовая выборка: {X_test.shape}")

Обучающая выборка: (102, 500, 500, 11)
Валидационная выборка: (22, 500, 500, 11)
Тестовая выборка: (23, 500, 500, 11)


In [22]:
import tensorflow as tf
from tensorflow.keras import layers, models, Input

def crop_to_match(encoder_layer, decoder_layer):
    # Получаем размеры по height и width
    ey, ex = encoder_layer.shape[1], encoder_layer.shape[2]
    dy, dx = decoder_layer.shape[1], decoder_layer.shape[2]
    cropy = int((ey - dy) // 2)
    cropx = int((ex - dx) // 2)
    if cropy > 0 or cropx > 0:
        encoder_layer = layers.Cropping2D(((cropy, cropy), (cropx, cropx)))(encoder_layer)
    return encoder_layer

def build_unet(input_shape=(128, 128, 3), num_classes=1):
    inputs = Input(shape=input_shape)

    # Downsampling
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2,2))(c3)

    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2,2))(c4)

    # Bottleneck
    c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3,3), activation='relu', padding='same')(c5)

    # Upsampling
    u6 = layers.Conv2DTranspose(512, (2,2), strides=(2,2), padding='same')(c5)
    c4_crop = crop_to_match(c4, u6)
    u6 = layers.concatenate([u6, c4_crop])
    c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3,3), activation='relu', padding='same')(c6)

    u7 = layers.Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(c6)
    c3_crop = crop_to_match(c3, u7)
    u7 = layers.concatenate([u7, c3_crop])
    c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3,3), activation='relu', padding='same')(c7)

    u8 = layers.Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(c7)
    c2_crop = crop_to_match(c2, u8)
    u8 = layers.concatenate([u8, c2_crop])
    c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3,3), activation='relu', padding='same')(c8)

    u9 = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(c8)
    c1_crop = crop_to_match(c1, u9)
    u9 = layers.concatenate([u9, c1_crop])
    c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c9)

    outputs = layers.Conv2D(num_classes, (1,1), activation='sigmoid')(c9)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model
input_shape = (500, 500, 11)
num_classes = 5
unet_model = build_unet(input_shape, num_classes)
unet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

unet_model.summary()

ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 124, 124, 256), (None, 125, 125, 256)]

In [ ]:
# Обучение модели
# epochs = 20 может быть большим для Google Colab, можно уменьшить, если есть ошибка памяти
history = unet_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=1,      # batch_size=1 подходит для очень больших изображений
    epochs=10,         # Можно изменить на большее значение для реального обучения
    verbose=1
)


In [30]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, Cropping2D
from tensorflow.keras.models import Model

def build_unet(input_shape, num_classes):
    inputs = Input(input_shape)

    # Энкодер с контролем размеров
    # Блок 1 (500x500 -> 250x250)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(2)(conv1)

    # Блок 2 (250x250 -> 125x125)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(2)(conv2)

    # Блок 3 (125x125 -> 62x62)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D((2,2), padding='valid')(conv3)

    # Центральный блок
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(pool3)
    conv4 = Conv2D(512, 3, activation='relu', padding='same')(conv4)

    # Декодер с коррекцией размеров
    # Блок 5 (62x62 -> 124x124)
    up5 = Conv2DTranspose(256, 2, strides=2, padding='same')(conv4)
    merge5 = concatenate([conv3, up5])
    conv5 = Conv2D(256, 3, activation='relu', padding='same')(merge5)

    # Блок 6 (124x124 -> 248x248)
    up6 = Conv2DTranspose(128, 2, strides=2, padding='same')(conv5)
    merge6 = concatenate([conv2, up6])
    conv6 = Conv2D(128, 3, activation='relu', padding='same')(merge6)

    # Блок 7 (248x248 -> 496x496)
    up7 = Conv2DTranspose(64, 2, strides=2, padding='same')(conv6)
    merge7 = concatenate([conv1, up7])
    conv7 = Conv2D(64, 3, activation='relu', padding='same')(merge7)

    # Финализация размеров
    outputs = Conv2D(num_classes, 1, activation='softmax')(conv7)
    outputs = Cropping2D(((2,2), (2,2)))(outputs)  # 500x500

    return Model(inputs, outputs)

# Инициализация
model = build_unet((500,500,11), 5)
model.compile(optimizer='adam', loss='categorical_crossentropy')

ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 125, 125, 256), (None, 124, 124, 256)]

In [ ]:
# Обучение модели
history = model.fit(
    X_train, y_train,
    batch_size=4,
    epochs=20,
    validation_data=(X_val, y_val),
    verbose=1
)

# Оценка модели на тестовых данных
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

# Визуализация результатов обучения
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.show()

# Предсказание на тестовом изображении
sample_idx = 0  # индекс изображения из тестовой выборки
sample_image = X_test[sample_idx]
sample_mask = y_test[sample_idx]

# Добавляем размерность батча (1, ...)
prediction = model.predict(sample_image[np.newaxis, ...])

# Визуализация
plt.figure(figsize=(18, 6))

# Исходное изображение (берем первые 3 канала для визуализации как RGB)
plt.subplot(1, 3, 1)
plt.imshow(sample_image[..., :3])  # Показываем только первые 3 канала
plt.title('Input Image')

# Истинная маска (берем класс с максимальной вероятностью)
plt.subplot(1, 3, 2)
plt.imshow(np.argmax(sample_mask, axis=-1))
plt.title('True Mask')

# Предсказанная маска
plt.subplot(1, 3, 3)
plt.imshow(np.argmax(prediction[0], axis=-1))
plt.title('Predicted Mask')
plt.show()